# Markowitz et al. (2023): Opto-DA claims, on dandiset 000559

Paper: doi:10.1038/s41586-022-05611-2; claims file `claims/markowitz-2023.yaml`, written from the
version of record. Dataset: DANDI:000559, version 0.261113.1911, the most recent published
version (the paper cites none); its release notes say earlier versions lacked the catch trials
and mislabelled the light source and groups, which the claims below need. Files are streamed.

This notebook tests the closed-loop Opto-DA claims (Fig. 3, Extended Data Fig. 10) on the
reinforcement experiments of the 20 DAT-IRES-cre::Ai32 mice and 12 no-opsin controls, with the
online MoSeq labels (Methods, Offline variant: the Opto-DA experiments of Figs. 3 and 4 use the
online variant).

| claim | here |
|---|---|
| fig3e-optoda-learning | tested |
| fig3f-cumulative-counts | tested |
| fig3e-learner-count | tested |
| fig3h-stim-post-correlation | tested |
| fig3i-optoda-entropy | tested |
| fig3l-no-kinematic-effect | tested |
| fig3m-long-stim-velocity | tested |
| ed10e-no-duration-change | tested |
| ed10i-no-place-preference | tested |
| fig3jk-post-predictable | tested |
| ed10-target-specific, ed10d-optoda-vigour | blocked, see the record |

In [ ]:
import zlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import markowitz as M

PAPER = "markowitz-2023"
def rng_for(claim, k=0):
    return np.random.default_rng(zlib.crc32(f"{PAPER}/{claim}".encode()) + k)

T0 = time.time()
A = M.list_assets()
rows = [r for _, r in A[A.kind.isin(["reinforcement", "velocity-modulation"])].iterrows()]
SESS = M.load_many(M.load_opto, rows)
print(len(SESS), "files loaded in", round(time.time() - T0), "s")

In [ ]:
OPTO = {"snc-dls-ai32", "snc-dls-ai32-jrcamp1a"}
CTRL = {"ctrl", "snc-dls-ctrl", "vta-nacc-ctrl"}
R = []
for i, d in enumerate(SESS):
    tg = d.get("ts_target_syllable", [])
    for k in range(len(tg)):
        R.append(dict(i=i, mouse=d["subject"], group=d["group"], kind="velmod" if "velocity-modulation" in d["path"] else "reinf",
                      target=int(tg[k]), sn=d["ts_session_number"][k], stim=d["ts_stim_duration"][k],
                      etype=d["ts_experiment_type"][k], exclude=bool(d["ts_exclude"][k]), start=d["start"],
                      n_det=len(d["det_time"]), n_stimdet=int(d["det_stim"].sum()), n_on=len(d["stim_on"]),
                      dur=d["t_last"] - d["t_first"]))
R = pd.DataFrame(R)
R["arm"] = np.where(R.group.isin(OPTO), "Opto-DA", np.where(R.group.isin(CTRL), "control", "other"))
R["day"] = R.start.map(lambda t: t.date())

In [ ]:
print("files by group and kind:"); print(R.drop_duplicates("i").groupby(["group", "kind"]).size().to_string())
print("\nmice per arm:", R.drop_duplicates("mouse").groupby("arm").size().to_dict())
print("\nrows (file x target) by experiment type and stimulation duration:")
print(R.groupby(["arm", "etype", "stim"], dropna=False).size().to_string())
print("\nsession numbers of 0.25-s reinforcement rows:", R[(R.stim == 0.25) & (R.kind == "reinf")].sn.value_counts().sort_index().to_dict())
print("excluded rows:", R[R.exclude][["mouse", "sn", "target"]].to_dict("records"))
st = R[R.sn.isin([1, 2]) & (R.kind == "reinf")].drop_duplicates("i")
print("\nstimulation files: detections per file median", st.n_det.median(), "; stimulated share",
      round(st.n_stimdet.sum() / st.n_det.sum(), 3), "(the Methods give 75%)")
print("stimulated detections equal light onsets in", (st.n_stimdet == st.n_on).sum(), "of", len(st), "files")
lat = []
for i in st.i:
    d = SESS[i]
    if len(d["stim_on"]) and d["det_stim"].sum():
        dt = np.subtract.outer(d["stim_on"], d["det_time"][d["det_stim"]])
        lat.append(np.median(np.min(np.abs(dt), 1)))
lat = np.array(lat)
print("per file, median time from a light onset to the nearest stimulated detection (s): 0 in",
      int(np.sum(lat < 1e-3)), "of", len(lat), "files; largest", lat.max().round(3))
print("experiment length (s): median", R.drop_duplicates("i").dur.median().round(1), "range",
      R.drop_duplicates("i").dur.min().round(1), "to", R.drop_duplicates("i").dur.max().round(1))
print("online labels sorted in time in every file:", all(d["syl_sorted"] for d in SESS))
# clock check: the online target detections must fall on instances of the target syllable
hit = []
for i in st.i[:200]:
    d = SESS[i]; inst = d["inst"]
    k = np.searchsorted(inst.onset.to_numpy(), d["det_time"], side="right") - 1
    ok = k >= 0
    lab = inst.label.to_numpy()[k[ok]]
    hit.append(np.mean(lab == d["det_target"][ok]))
print("share of detections that fall inside an instance of the target label: median", np.median(hit).round(3))
for sh in [-30, -10, 10, 30]:
    hh = []
    for i in st.i[:50]:
        d = SESS[i]; inst = d["inst"]
        k = np.searchsorted(inst.onset.to_numpy(), d["det_time"] + sh / 30, side="right") - 1
        ok = k >= 0
        hh.append(np.mean(inst.label.to_numpy()[k[ok]] == d["det_target"][ok]))
    print(f"   with detections shifted by {sh} frames: {np.median(hh):.3f}")
vm = pd.DataFrame([dict(mouse=d["subject"], description=d["session_description"], seconds=round(d["t_last"] - d["t_first"]),
                        stimulated=int(d["det_stim"].sum())) for d in SESS if "velocity-modulation" in d["path"]])
vm["direction"] = vm.description.str.extract(r"-(up|down)")[0].fillna("not stated")
print("\nvelocity-modulation files by mouse and the direction their session description states:")
print(vm.groupby(["mouse", "direction"]).size().unstack(fill_value=0).to_string())
v = pd.concat([d["inst"] for d in SESS[:100] if "vel" in d["inst"]])
print("instance mean 2D speed (mm/s) percentiles 1, 50, 99:", np.nanpercentile(v.vel, [1, 50, 99]).round(1))

In [ ]:
# Target counts: run-length instances of the target in the first 30 min of each experiment.
def tcount(i, target, tmax=1800):
    d = SESS[i]; inst = d["inst"]
    t = inst.onset.to_numpy() - d["t_first"]
    return int(np.sum((inst.label.to_numpy() == target) & (t < tmax)))

def target_table(stim_dur=0.25, etypes=("reinforcement",), tcount=tcount):
    rr = R[(R.kind == "reinf") & (R.stim == stim_dur) & R.etype.isin(etypes) & R.arm.isin(["Opto-DA", "control"]) & ~R.exclude]
    out = []
    for (m, tg), g in rr.groupby(["mouse", "target"]):
        sti = g[g.sn.isin([1, 2])]
        if sti.sn.nunique() < 2:
            continue
        stim_day = sti.day.min()
        base = g[g.sn.isin([-1, 0]) & (g.day < stim_day)]
        if len(base) == 0:
            continue
        bday = base.day.max()                          # most recent previous baseline day
        base = base[base.day == bday]
        post = g[g.sn.isin([3, 4])]
        c_stim = sum(tcount(i, tg) for i in sti.i)
        c_base = np.mean([tcount(i, tg) for i in base.i]) * 2
        c_post = sum(tcount(i, tg) for i in post.i) if post.sn.nunique() == 2 else np.nan
        out.append(dict(mouse=m, arm=g.arm.iloc[0], target=tg, stim=c_stim, base=c_base, post=c_post,
                        n_base=len(base), days_before=(stim_day - bday).days,
                        base_i=list(base.i), post_i=list(post.i), stim_i=list(sti.i)))
    T = pd.DataFrame(out)
    T["lfc"] = np.log2((T.stim + 2) / (T.base + 2))
    T["dc"] = T.stim - T.base
    T["dc_post"] = T.post - T.base
    ntg = T.groupby("mouse").target.transform("size")
    return T[ntg >= 2]

T3 = target_table()
PMS = T3.groupby(["mouse", "arm"]).agg(lfc=("lfc", "mean"), dc=("dc", "mean"), dc_post=("dc_post", "mean"),
                                         n_targets=("target", "size")).reset_index()
print("mouse-target pairs:", len(T3), "; mice per arm:", PMS.groupby("arm").size().to_dict())
print("targets per mouse:", PMS.n_targets.value_counts().to_dict(), "; baseline experiments per pair:",
      T3.n_base.value_counts().to_dict(), "; days from baseline to stimulation:", T3.days_before.value_counts().to_dict())

def mwu(a, b, alt):
    u = stats.mannwhitneyu(a, b, alternative=alt)
    return u.statistic, u.statistic / (len(a) * len(b)), u.pvalue

## fig3e-optoda-learning

In [ ]:
a = PMS[PMS.arm == "Opto-DA"].lfc; b = PMS[PMS.arm == "control"].lfc
U, f, p = mwu(a, b, "greater")
print(f"log2 fold change per mouse: Opto-DA median {a.median():.3f} (n={len(a)}), control median {b.median():.3f} (n={len(b)})")
print(f"one-sided Mann-Whitney U = {U:.0f}, f = {f:.2f}, P = {p:.4f}")
print(PMS.sort_values("lfc").round(3).to_string(index=False))
# variant: counts from the offline labels of the same experiments (whole files)
def tcount_off(i, target):
    oi = SESS[i].get("off_inst")
    return int((oi.label == target).sum()) if oi is not None else np.nan
T3off = target_table(tcount=lambda i, target: tcount_off(i, target))
PMoff = T3off.groupby(["mouse", "arm"]).agg(lfc=("lfc", "mean"), dc=("dc", "mean")).reset_index()
for col in ["lfc", "dc"]:
    U, f, p = mwu(PMoff[PMoff.arm == "Opto-DA"][col], PMoff[PMoff.arm == "control"][col], "greater")
    print(f"variant, offline labels, {col}: U = {U:.0f}, f = {f:.2f}, P = {p:.4f}")

**Result: not reproduced.** The log2 fold change in target counts is not significantly larger in the 20 Opto-DA mice than in the 12 controls (medians 0.215 and 0.175; one-sided U = 152, f = 0.63, P = 0.110); with offline labels P = 0.370.

## fig3f-cumulative-counts and fig3e-learner-count

In [ ]:
a = PMS[PMS.arm == "Opto-DA"].dc; b = PMS[PMS.arm == "control"].dc
U, f, p = mwu(a, b, "greater")
print(f"change in counts per mouse: Opto-DA median {a.median():.1f}, control median {b.median():.1f}; U = {U:.0f}, f = {f:.2f}, P = {p:.4f}")

**Result: not reproduced.** The change in counts above baseline is larger in Opto-DA mice than in controls, but not significantly (medians 32.2 and 23.2; U = 160, f = 0.67, P = 0.062).

In [ ]:
thr = PMS[PMS.arm == "control"].dc.max()
LEARN = PMS[(PMS.arm == "Opto-DA") & (PMS.dc > thr)].mouse.tolist()
print(f"largest mean change among controls: {thr:.1f}; Opto-DA mice above it: {len(LEARN)} of {(PMS.arm == 'Opto-DA').sum()}")
print("learners:", LEARN)

**Result: not reproduced.** 7 of 20 Opto-DA mice exceed the largest mean change among controls (60.5), where the paper reports 9.

## fig3h-stim-post-correlation

In [ ]:
rng = rng_for("fig3h-stim-post-correlation")
for name, sub in [("learners", PMS[PMS.mouse.isin(LEARN)]), ("controls", PMS[PMS.arm == "control"])]:
    sub = sub.dropna(subset=["dc", "dc_post"])
    r = np.corrcoef(sub.dc, sub.dc_post)[0, 1]
    null = np.array([np.corrcoef(rng.permutation(sub.dc.to_numpy()), sub.dc_post)[0, 1] for _ in range(1000)])
    p = (1 + np.sum(null >= r)) / 1001
    print(f"{name}: n = {len(sub)}, Pearson r = {r:.3f}, one-sided shuffle P = {p:.4f}")
seeds_r = []
for k in [1000, 2000, 3000, 4000]:
    rg = rng_for("fig3h-stim-post-correlation", k)
    sub = PMS[PMS.mouse.isin(LEARN)].dropna(subset=["dc", "dc_post"])
    r = np.corrcoef(sub.dc, sub.dc_post)[0, 1]
    null = np.array([np.corrcoef(rg.permutation(sub.dc.to_numpy()), sub.dc_post)[0, 1] for _ in range(1000)])
    seeds_r.append((1 + np.sum(null >= r)) / 1001)
print("learners' P under seeds +1000..+4000:", np.round(seeds_r, 4))
seeds_c = []
for k in [1000, 2000, 3000, 4000]:
    rg = rng_for("fig3h-stim-post-correlation", k)
    sub = PMS[PMS.arm == "control"].dropna(subset=["dc", "dc_post"])
    r = np.corrcoef(sub.dc, sub.dc_post)[0, 1]
    null = np.array([np.corrcoef(rg.permutation(sub.dc.to_numpy()), sub.dc_post)[0, 1] for _ in range(1000)])
    seeds_c.append((1 + np.sum(null >= r)) / 1001)
print("controls' P under seeds +1000..+4000:", np.round(seeds_c, 4))

**Result: partially reproduced.** In the 7 learners the change during stimulation predicts the change after it (r = 0.868, within 0.89 +- 0.089; one-sided P = 0.031, and from 0.025 to 0.038 under the further seeds), but the controls' correlation is also significant (r = 0.633, P = 0.025, from 0.030 to 0.041 under the further seeds), where the paper reports P = 0.082.

## Trial-aligned measures for fig3i, fig3l, fig3m and ed10e
Stimulation trials are online detections with light; catch trials detections without
(TargetDetections). Each trial is aligned to the instance it falls in.

In [ ]:
NB = 10                                               # five-syllable bins up to 50 syllables
def trial_measures(i):
    d = SESS[i]; inst = d["inst"]
    on = inst.onset.to_numpy(); lab = inst.label.to_numpy()
    k = np.searchsorted(on, d["det_time"], side="right") - 1
    rows_ = []
    vars_ = {v: inst[v].to_numpy() for v in ["vel", "angvel", "hvel", "acc"] if v in inst}
    if "angvel" in vars_:
        vars_["angvel"] = np.abs(vars_["angvel"])
    for kk, stim_ in zip(k, d["det_stim"]):
        if kk - 5 < 0 or kk + 5 * NB >= len(lab):
            continue
        r = dict(stim=bool(stim_), k=kk, dur=inst.dur.iloc[kk])
        for v, x in vars_.items():
            pre = np.nanmean(x[kk - 5:kk])
            for b in range(NB):
                r[f"{v}_{b}"] = np.nanmean(x[kk + 5 * b:kk + 5 * b + 5]) - pre
                r[f"{v}raw_{b}"] = np.nanmean(x[kk + 5 * b:kk + 5 * b + 5])
        rows_.append(r)
    return pd.DataFrame(rows_), lab

def cond_entropy(seqs, K):
    """Conditional entropy (bits) of transitions pooled over sequences, Perks prior 1/K."""
    C = np.zeros((K, K))
    for s in seqs:
        if len(s) > 1:
            np.add.at(C, (s[:-1], s[1:]), 1)
    rows_ = C.sum(1) > 0
    Cp = C[rows_] + 1 / K
    P = Cp / Cp.sum(1, keepdims=True)
    w = C[rows_].sum(1) / C.sum()
    return float(np.sum(w * -(P * np.log2(P)).sum(1)))

def experiment_table(rr):
    out = []
    for _, r in rr.drop_duplicates("i").iterrows():
        tm, lab = trial_measures(r.i)
        if len(tm) == 0 or tm.stim.sum() < 1 or (~tm.stim).sum() < 1:
            continue
        K = lab.max() + 1
        for cond, g in tm.groupby("stim"):
            row = dict(i=r.i, mouse=r.mouse, arm=r.arm, sn=r.sn, target=r.target, stim=cond, n=len(g), dur=g.dur.mean())
            row.update(g.drop(columns=["stim", "k", "dur"]).mean().to_dict())
            ks = g.k.to_numpy()
            pre = cond_entropy([lab[k - 5:k] for k in ks], K)
            for b in range(NB):
                row[f"ent_{b}"] = cond_entropy([lab[k + 5 * b:k + 5 * b + 5] for k in ks], K) - pre
            out.append(row)
    return pd.DataFrame(out)

t0 = time.time()
E25 = experiment_table(R[(R.kind == "reinf") & (R.stim == 0.25) & (R.etype == "reinforcement") & R.sn.isin([1, 2])
                         & R.arm.isin(["Opto-DA", "control"]) & ~R.exclude])
E3 = experiment_table(R[(R.kind == "reinf") & (R.stim == 3.0) & R.sn.isin([1, 2]) & R.arm.isin(["Opto-DA", "control"]) & ~R.exclude])
print("250-ms experiments with both trial types:", E25.i.nunique(), "; 3-s pulsed:", E3.i.nunique(), "; in", round(time.time() - t0), "s")
print("3-s pulsed experiments by arm and mice:", E3.drop_duplicates("i").groupby("arm").mouse.nunique().to_dict())

def catch_z(E, col):
    """Values z-scored with the mean and s.d. of catch trials (Methods), per arm."""
    c = E[~E.stim][col]
    return (E[col] - c.mean()) / c.std()

def stim_vs_catch(E, var, alt, nb=NB):
    res = {}
    for arm, g in E.groupby("arm"):
        ps, us = [], []
        for b in range(nb):
            col = f"{var}_{b}"
            z = catch_z(g, col)
            s, c = z[g.stim].dropna(), z[~g.stim].dropna()
            u = stats.mannwhitneyu(s, c, alternative=alt)
            ps.append(u.pvalue); us.append(u.statistic / (len(s) * len(c)))
        res[arm] = (np.array(ps), M.holm(ps), np.array(us))
    return res

## fig3i-optoda-entropy

In [ ]:
res = stim_vs_catch(E25, "ent", "two-sided")
for arm, (p, ph, f) in res.items():
    g = E25[E25.arm == arm]
    med = [(g[g.stim][f"ent_{b}"].median() - g[~g.stim][f"ent_{b}"].median()) for b in range(NB)]
    print(f"{arm}: experiments {g.i.nunique()}; stim minus catch, median entropy change per bin {np.round(med, 3).tolist()}")
    print(f"   f per bin {np.round(f, 2).tolist()}")
    print(f"   P (Holm) per bin {np.round(ph, 4).tolist()}")

**Result: reproduced.** In Opto-DA mice, entropy after stimulation is higher than after catch trials in the first five-syllable bin (median difference 0.07 bits, f = 0.57, Holm P = 0.017) and in every later bin; in controls no bin is significant (smallest Holm P 0.22).

## fig3l-no-kinematic-effect

In [ ]:
anysig = False
for var in ["vel", "angvel", "hvel", "acc"]:
    res = stim_vs_catch(E25, var, "greater")
    for arm, (p, ph, f) in res.items():
        print(f"{var}, {arm}: min P (Holm) {ph.min():.4f} at bin {ph.argmin()}; f range {f.min():.2f} to {f.max():.2f}")
        if arm == "Opto-DA" and (ph < 0.05).any():
            anysig = True
print("any Opto-DA bin significant (Methods form):", anysig)
# variant: the notebook's values without pre-onset subtraction
for var in ["vel", "angvel", "hvel", "acc"]:
    res = stim_vs_catch(E25, var + "raw", "greater")
    print(f"variant without pre-onset subtraction, {var}: Opto-DA min P (Holm) {res['Opto-DA'][1].min():.4f}; control {res['control'][1].min():.4f}")
# variant: per-mouse tests (the legend's "in any of the mice")
sigm = []
for m, g in E25[E25.arm == "Opto-DA"].groupby("mouse"):
    if g.stim.sum() < 3:
        continue
    for var in ["vel", "angvel", "hvel", "acc"]:
        r_ = stim_vs_catch(g, var, "greater")
        if (r_["Opto-DA"][1] < 0.05).any():
            sigm.append((m, var))
print("per-mouse variant, Opto-DA mice and variables with a significant bin:", sigm)

**Result: reproduced.** No bin of 2D, angular or height velocity or acceleration differs between stimulation and catch trials in Opto-DA mice (smallest Holm P 1.0 in the Methods' form); the variants without pre-onset subtraction and per mouse find none either.

## fig3m-long-stim-velocity

In [ ]:
res = stim_vs_catch(E3, "vel", "greater")
for arm, (p, ph, f) in res.items():
    g = E3[E3.arm == arm]
    print(f"{arm}: {g.i.nunique()} experiments from {g.mouse.nunique()} mice; P (Holm) per bin {np.round(ph, 4).tolist()}; f {np.round(f, 2).tolist()}")
res_raw = stim_vs_catch(E3, "velraw", "greater")
print("variant without pre-onset subtraction:", {a: np.round(v[1], 4).tolist() for a, v in res_raw.items()})

**Result: not reproduced.** In the 36 3-s pulsed experiments of the 3 Opto-DA mice, 2D velocity after stimulation does not exceed catch trials in any bin (Holm P 1.0 in every bin).

## fig3jk-post-predictable
"In the experiments after stimulation, sequences around the target become more predictable; the
most likely transitions into and out of the target become more likely."

In [ ]:
def trans_counts(idx, tg, inbound):
    c = {}
    for i in idx:
        lab = SESS[i]["inst"].label.to_numpy()
        if inbound:
            other = lab[:-1][lab[1:] == tg]
        else:
            other = lab[1:][lab[:-1] == tg]
        for k, n in zip(*np.unique(other, return_counts=True)):
            c[k] = c.get(k, 0) + n
    return c

NR = 10                                             # ranks kept (a standard choice; the top rank decides)
def rank_rows(day_col, inbound):
    rows_ = []
    for _, r in T3.iterrows():
        who = "learner" if r.mouse in LEARN else ("control" if r.arm == "control" else None)
        if who is None or len(r[day_col]) == 0:
            continue
        b = trans_counts(r.base_i, r.target, inbound)
        order = sorted(b, key=lambda k: -b[k])[:NR]
        if len(order) < NR:
            continue
        p = trans_counts(r[day_col], r.target, inbound)
        x = np.array([p.get(k, 0) + 1 for k in order], float)
        x = pd.Series(x).rolling(5, center=True, min_periods=1).mean().to_numpy()
        rows_.append((who, x))
    return rows_

rng = rng_for("fig3jk-post-predictable")
for day_col, label in [("post_i", "post-stimulation day"), ("stim_i", "variant, stimulation day")]:
    for inbound in [True, False]:
        rr_ = rank_rows(day_col, inbound)
        L_ = np.array([x for w, x in rr_ if w == "learner"]); C_ = np.array([x for w, x in rr_ if w == "control"])
        ratio = np.log2(L_.mean(0) / C_.mean(0))
        boot = np.array([np.log2(L_[rng.integers(0, len(L_), len(L_))].mean(0)[0] / C_[rng.integers(0, len(C_), len(C_))].mean(0)[0])
                         for _ in range(1000)])
        lo, hi = np.percentile(boot, [2.5, 97.5])
        print(f"{label}, {'inbound' if inbound else 'outbound'}: {len(L_)} learner and {len(C_)} control pairs; "
              f"log2 ratio at the top rank {ratio[0]:.3f} (95% bootstrap {lo:.3f} to {hi:.3f}); ranks 1-5 {np.round(ratio[:5], 3).tolist()}")

**Result: partially reproduced.** On the post-stimulation day the top-ranked inbound and outbound transitions are more frequent in learners than in controls (log2 ratios 0.374 and 0.234), but both 95% bootstrap intervals include 0 (-0.048 to 0.773 and -0.127 to 0.546). On the stimulation day, the Fig. 3k legend's reading, both intervals exclude 0 (0.443 and 0.508).

## ed10e-no-duration-change

In [ ]:
D = E25.pivot_table(index=["mouse", "arm", "target", "sn"], columns="stim", values="dur").dropna().reset_index()
D["diff"] = D[True] - D[False]
D["who"] = np.where(D.mouse.isin(LEARN), "learner", np.where(D.arm == "control", "control", "non-learner"))
ps = []
for sn in [1, 2]:
    a = D[(D.who == "learner") & (D.sn == sn)]["diff"]; b = D[(D.who == "control") & (D.sn == sn)]["diff"]
    U, f, p = mwu(a, b, "two-sided"); ps.append(p)
    print(f"stimulation experiment {sn}: learner pairs {len(a)}, control pairs {len(b)}; median stim - catch duration "
          f"{a.median()*1000:.1f} ms against {b.median()*1000:.1f} ms; U = {U:.0f}, f = {f:.2f}, P = {p:.3f}")
print("Holm-corrected:", np.round(M.holm(ps), 3))

**Result: reproduced.** The stimulated minus catch duration does not differ between learners and controls in either stimulation experiment (P = 0.941 and 0.557; U = 1525 and 1352).

## ed10i-no-place-preference

In [ ]:
def jsd(p, q):
    p = p / p.sum(); q = q / q.sum(); m_ = (p + q) / 2
    kl = lambda a, b: np.sum(np.where(a > 0, a * np.log2(np.where(a > 0, a, 1) / np.where(b > 0, b, 1)), 0))
    return 0.5 * kl(p, m_) + 0.5 * kl(q, m_)

rr = R[(R.kind == "reinf") & (R.stim == 0.25) & (R.etype == "reinforcement") & R.arm.isin(["Opto-DA", "control"]) & ~R.exclude]
allxy = pd.concat([SESS[i]["inst"][["x", "y"]] for i in rr.i.unique()])
ex = np.nanpercentile(allxy.x, [0.5, 99.5]); ey = np.nanpercentile(allxy.y, [0.5, 99.5])
EX = np.linspace(*ex, 11); EY = np.linspace(*ey, 11)
J = []
for (m, tg), g in rr.groupby(["mouse", "target"]):
    st_ = g[g.sn.isin([1, 2])]; ot = g[~g.sn.isin([1, 2])]
    if len(st_) == 0 or len(ot) == 0:
        continue
    def hist(idx, shared=True):
        xy = pd.concat([SESS[i]["inst"].query("label == @tg")[["x", "y"]] for i in idx]).dropna()
        if shared:
            return np.histogram2d(xy.x, xy.y, [EX, EY])[0]
        return np.histogram2d(xy.x, xy.y, 10)[0]
    J.append(dict(mouse=m, arm=g.arm.iloc[0], target=tg, jsd=jsd(hist(st_.i.unique()), hist(ot.i.unique())),
                  jsd_own=jsd(hist(st_.i.unique(), False), hist(ot.i.unique(), False))))
J = pd.DataFrame(J)
for col, name in [("jsd", "shared grid"), ("jsd_own", "variant, own bin edges")]:
    a = J[J.arm == "Opto-DA"][col]; b = J[J.arm == "control"][col]
    U, f, p = mwu(a, b, "two-sided")
    print(f"target performance, {name}: pairs {len(a)} + {len(b)} = {len(J)}; median JSD {a.median():.3f} against {b.median():.3f}; U = {U:.0f}, f = {f:.2f}, P = {p:.3f}")
    pm_ = J.groupby(["mouse", "arm"])[col].mean().reset_index()
    a = pm_[pm_.arm == "Opto-DA"][col]; b = pm_[pm_.arm == "control"][col]
    U, f, p = mwu(a, b, "two-sided")
    print(f"   by mouse: {len(a)} + {len(b)} mice; U = {U:.0f}, f = {f:.2f}, P = {p:.3f}")
# experiment-wide: all instance onsets on stimulation days against other days, per mouse
W = []
for m, g in rr.groupby("mouse"):
    hs = lambda idx: np.histogram2d(*pd.concat([SESS[i]["inst"][["x", "y"]] for i in idx]).dropna().to_numpy().T, [EX, EY])[0]
    st_ = g[g.sn.isin([1, 2])].i.unique(); ot = g[~g.sn.isin([1, 2])].i.unique()
    W.append(dict(mouse=m, arm=g.arm.iloc[0], jsd=jsd(hs(st_), hs(ot))))
W = pd.DataFrame(W)
a = W[W.arm == "Opto-DA"].jsd; b = W[W.arm == "control"].jsd
U, f, p = mwu(a, b, "two-sided")
print(f"experiment-wide, by mouse: {len(a)} + {len(b)} mice; U = {U:.0f}, f = {f:.2f}, P = {p:.3f}")

**Result: reproduced on the criterion's tests.** The Jensen-Shannon divergence of target-onset locations between stimulation and other days does not differ between Opto-DA and control pairs (U = 4053, P = 0.309) or mice (U = 103, P = 0.521). The statement's experiment-wide comparison does not hold: by mouse, Opto-DA divergences are lower than controls' (U = 48, f = 0.20, P = 0.005).

## ed2a-syllable-repertoire, reading (b)
The Opto-DA experiments' offline labels (reinforcement files); the photometry experiments are in
`photometry.ipynb`.

In [ ]:
OFF = pd.concat([d["off_inst"].assign(mouse=d["subject"]) for d in SESS if d.get("off_inst") is not None])
dur = OFF.dur.dropna() * 1000
print(f"Opto-DA experiments with offline labels: {sum(d.get('off_inst') is not None for d in SESS)}, mice {OFF.mouse.nunique()}, "
      f"instances {len(OFF)}: median {dur.median():.0f} ms, mean {dur.mean():.0f} ms, s.d. {dur.std():.0f} ms")
use_o = OFF.groupby("mouse").label.value_counts(normalize=True).unstack(fill_value=0).mean(0)
print("syllables with mean usage over these mice > 1%:", int((use_o > 0.01).sum()))

**Result: reading (b) only in part, not decided here.** The offline labels of the 1,340 Opto-DA experiments give a median duration of 467 ms and a mean of 651 ms, outside the tolerances; 35 syllables pass 1%. The claim is decided in `photometry.ipynb` under reading (a).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| claim | outcome |
|---|---|
| fig3e-optoda-learning | not reproduced (P = 0.110) |
| fig3f-cumulative-counts | not reproduced (P = 0.062) |
| fig3e-learner-count | not reproduced (7 learners) |
| fig3h-stim-post-correlation | partially reproduced (controls also significant) |
| fig3i-optoda-entropy | reproduced |
| fig3l-no-kinematic-effect | reproduced |
| fig3m-long-stim-velocity | not reproduced |
| ed10e-no-duration-change | reproduced |
| ed10i-no-place-preference | reproduced on target-onset locations |
| fig3jk-post-predictable | partially reproduced (post-day intervals include 0) |
| ed10d-optoda-vigour | blocked (direction of the velocity-modulation experiments stated for 4 of 9 mice) |
| ed10-target-specific | blocked (not run in the time) |

How the analysis differs from the paper's: target counts are run-length instances of the online
labels in the first 30 min of each experiment; the baseline is the target's own -1 and 0 sessions
on the most recent day before stimulation (the TargetSessions table); trial measures align each
online detection to the instance it falls in; sequence entropy is the conditional entropy of
pooled transitions with a Perks prior; the place-preference grid spans the 0.5 to 99.5
percentiles of all positions.

What was found about the dataset: the OptogeneticSeries description says onsets are recorded as 1,
but the values are the power, 0.01 W; in 60 of 516 stimulation files the median time from a light onset
to the nearest stimulated detection is not 0 (up to 1.48 s); the stimulated share of detections is 0.707 where the
Methods give 75%; the online target detections fall inside an instance of the target label in a
median 99.7% of cases, and in 7 to 40% with the detections shifted, consistent with a shared clock;
the velocity-modulation files state the direction of stimulation (up or down) only in the session
descriptions of mice 5891 to 5894, and not for the five dls-ai32jr mice.